# CS 3892 / 5892 — Session 5 · Sets, Logic, and Solvers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-09-10-sets-and-propositional-logic.ipynb)

**Thursday, September 10, 2026.** Every example the slides showed, in both
SMT-LIB and Python, runnable here with nothing installed.

This notebook does **not** contain copies of the code. It runs the real files in
[`sessions/cs3892-2026-09-10-sets-and-propositional-logic/`](https://github.com/ttj/cs3892-examples/tree/main/sessions/cs3892-2026-09-10-sets-and-propositional-logic) —
the same files CI checks — so the notebook and the repository cannot drift apart.

| | |
|---|---|
| `smt2/` | SMT-LIB 2, the standard input language every SMT solver accepts |
| `python/` | the same thing through Z3's Python API, which is friendlier to write |

Each `.smt2` file carries a `; EXPECT:` line and each `.py` asserts its own
result, so a wrong answer is a failure here and in CI, not a number you have to
notice.

## Setup

Run this once. On Colab it clones the repo and installs Z3 (a few seconds); anywhere the repo already exists it is a no-op.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, define helpers -------
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-09-10-sets-and-propositional-logic"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

SM = ROOT / "sessions" / SESSION / "smt2"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    print(f"--- {pathlib.Path(path).name} " + "-" * max(0, 60 - len(pathlib.Path(path).name)))
    print(pathlib.Path(path).read_text().rstrip())
    print()

def run(path, show_source=True):
    """Run one example and stream its output. Raises if it does not pass.

    .smt2 goes through scripts/run_smt2.py, which checks the file's own
    `; EXPECT:` contract. .py is executed directly and asserts internally.
    The pip wheel for Z3 ships no `z3` CLI, which is why .smt2 is run through
    the Python bindings rather than a shell command -- identical everywhere.
    """
    path = pathlib.Path(path)
    if show_source:
        show(path)
    cmd = ([sys.executable, "scripts/run_smt2.py", str(path)] if path.suffix == ".smt2"
           else [sys.executable, str(path)])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.stderr.strip():
        print(r.stderr.rstrip(), file=sys.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

## 1. The policy, encoded — propositional logic

Slide 27. An employee may take extended leave if employed at least twelve months with unused leave remaining; contractors are never eligible. The model has told a contractor with eleven months' service that they *are* eligible.

Everything here is a plain boolean. **`unsat` is a proof**: no world satisfies the policy and the claim at once.

In [ ]:
run(SM  / "01_policy_booleans.smt2")
run(PYD / "01_policy_booleans.py")

## 2. One atom becomes arithmetic — integers

Slide 29. `e` was an opaque boolean; `months >= 12` is a **constraint**. Nobody tells Z3 that 11 is less than 12 — the `Int` sort carries a theory of arithmetic with it.

The second file changes 11 to 12 and the answer flips to `sat`, **with a model** — the world in which the claim holds.

In [ ]:
run(SM  / "02_policy_integers.smt2")
run(PYD / "02_policy_integers.py")

run(SM  / "03_policy_integers_sat.smt2")
run(PYD / "03_policy_integers_sat.py")

## 3. The theory is not decoration — integers vs reals

Slide 30. The same formula, \\(2x = 1\\), asked twice. **One word changes and the answer reverses.**

This is why *“is it satisfiable?”* is not a well-formed question until you say **satisfiable in what**. That qualifier is the *modulo* in *satisfiability modulo theories*.

In [ ]:
run(SM  / "04_half_int.smt2")
run(PYD / "04_half_int.py")

run(SM  / "05_half_real.smt2")
run(PYD / "05_half_real.py")

## 4. Arithmetic your machine does not do — bit-vectors

Slide 31. Is \\(x + 1 > x\\) ever false? In mathematics, no.

**Predict the answer before you run this.** A `BitVec` is a fixed-width machine word with wraparound, not a number — and the solver returns the exact value where it breaks. This is the class of bug that destroyed Ariane 5 flight 501 in 1996.

In [ ]:
run(SM  / "06_bitvector_overflow.smt2")
run(PYD / "06_bitvector_overflow.py")

## 5. Reasoning about code you cannot see — uninterpreted functions

Slide 32. Z3 has never seen `f`: no body, no definition, no name it recognises. It knows exactly one thing — **`f` is a function**, so equal inputs give equal outputs. That alone refutes the claim.

This is how a verifier reasons about a library call it has no source for.

In [ ]:
run(SM  / "07_uninterpreted_function.smt2")
run(PYD / "07_uninterpreted_function.py")

## 6. The identity every verifier is built on

Slide 23. \\(\\varphi\\) is **valid** exactly when \\(\\neg\\varphi\\) is **unsatisfiable**.

A solver has no *“is it valid?”* button. To show a formula is a tautology you assert its **negation** and watch the solver fail. Every verifier in this course is built on this one move.

In [ ]:
run(SM  / "08_validity_by_refutation.smt2")
run(PYD / "08_validity_by_refutation.py")

## 7. Now break them

The examples above are the floor, not the ceiling. Some things worth trying —
each is a one-line edit in the cell below.

1. In the policy, delete the contractor assertion. Does it become satisfiable? Who is the model?
2. Make the bit-vector 16 bits wide instead of 8. Predict the counterexample before you run it.
3. Use `bvugt` (unsigned) instead of `bvsgt` (signed) in the overflow example. Where does it break now, and why is it a different number?
4. Give `f` a second argument, or assert `f(a) == a` as well. What does the solver do?
5. Ask for something genuinely hard: `x*x == 2` over `Real`, then over `Int`.

Anything you write here is scratch — the files on disk are untouched.

In [ ]:
from z3 import *

# Scratch. Try one of the exercises above, or anything else.
x = Real("x")
s = Solver()
s.add(x * x == 2)
print(s.check())
if s.check() == sat:
    print(s.model())          # an algebraic number, printed as a root object

## Where this goes next

| When | What |
|---|---|
| **Tue Sep 15** | First-order logic — quantifiers, and the duality between a formula and the set of states satisfying it |
| **Thu Sep 17** | SAT — how the search actually works: CNF, DPLL/CDCL, resolution, UNSAT cores |
| **Tue Sep 22** | SMT proper — DPLL(T), the theory solvers, bounded reachability |
| **HW1** | Z3 from Python, on all of the above |

Everything here also runs from a terminal:

```bash
git clone https://github.com/ttj/cs3892-examples.git
cd cs3892-examples
pip install z3-solver
bash scripts/check_examples.sh
```

Source: [`sessions/cs3892-2026-09-10-sets-and-propositional-logic/`](https://github.com/ttj/cs3892-examples/tree/main/sessions/cs3892-2026-09-10-sets-and-propositional-logic)